# Task 1: Basic Network Sniffer Tutorial
This notebook explains step-by-step how to build a basic network packet sniffer using Python and the `scapy` library.

### What is a Packet Sniffer?
A packet sniffer (or network analyzer) captures network traffic as it passes over the network interface. It intercept and parses raw packets, showing protocols, source/destination IP addresses, and payloads.

### Prerequisites
- Python 3.x
- Administrative/Root Privileges (Sniffing requires access to raw sockets)
- Windows users: Install **Npcap** or WinPcap driver to enable packet capturing.

## Step 1: Importing Required Libraries
We will use `scapy.all` to import the necessary tools:
- `sniff`: The function that listens to and captures packets.
- `IP`: The IP protocol layer representation.
- `TCP` and `UDP`: The transport layer protocol representation.
- `Raw`: The raw payload carrier layer.
- `conf`: System configuration values, useful to read the default active interface.

In [ ]:
import sys
from scapy.all import sniff, IP, TCP, UDP, Raw, conf

print(f"[*] Active Network Interface detected: {conf.iface}")

## Step 2: Defining the Packet Parsing Callback Function
We need a function that will execute for every single packet captured. This function will check:
1. If the packet contains an IP layer (Layer 3).
2. The source and destination IP addresses.
3. The Layer 4 protocol (TCP, UDP, ICMP, etc.) and port numbers.
4. If there's any Raw data (payload) and try to decode it.

In [ ]:
def packet_callback(packet):
    # Check if the packet has an IP (Layer 3) layer
    if packet.haslayer(IP):
        src_ip = packet[IP].src
        dst_ip = packet[IP].dst
        proto = packet[IP].proto
        
        proto_name = "Unknown"
        payload_desc = ""
        
        # Check if the packet has a TCP layer
        if packet.haslayer(TCP):
            sport = packet[TCP].sport
            dport = packet[TCP].dport
            proto_name = f"TCP ({sport} -> {dport})"
            
            # Try to read raw data
            if packet.haslayer(Raw):
                raw_data = packet[Raw].load
                try:
                    # Decode as ASCII/UTF-8 and truncate for display
                    payload_desc = raw_data.decode('utf-8', errors='ignore')[:100].strip()
                    payload_desc = f" | Payload: {payload_desc}"
                except:
                    payload_desc = f" | Raw Payload ({len(raw_data)} bytes)"
                    
        # Check if the packet has a UDP layer
        elif packet.haslayer(UDP):
            sport = packet[UDP].sport
            dport = packet[UDP].dport
            proto_name = f"UDP ({sport} -> {dport})"
            
            if packet.haslayer(Raw):
                raw_data = packet[Raw].load
                try:
                    payload_desc = raw_data.decode('utf-8', errors='ignore')[:100].strip()
                    payload_desc = f" | Payload: {payload_desc}"
                except:
                    payload_desc = f" | Raw Payload ({len(raw_data)} bytes)"
                    
        # Check for ICMP (e.g. Ping)
        elif proto == 1:
            proto_name = "ICMP"
            
        # Display captured traffic formatted: IP source -> IP destination
        print(f"[IP] {src_ip} -> {dst_ip} | Protocol: {proto_name}{payload_desc}")

## Step 3: Running the Sniffer
Now we call the `sniff()` function from Scapy. We pass:
- `prn=packet_callback`: The function to run for every packet.
- `store=0`: Instructs Scapy not to keep captured packets in memory, avoiding memory exhaustion.
- `count=10`: Let's capture exactly 10 packets for demonstration, then stop.

**Note:** If you run this cell and receive a permission error, make sure you are running the Jupyter server/notebook environment as an **Administrator**.

In [ ]:
print("[*] Sniffing 10 packets... Generate network traffic (e.g., ping or open a website) to capture them.")
try:
    sniff(prn=packet_callback, store=0, count=10)
    print("[*] Finished capturing 10 packets.")
except PermissionError:
    print("[!] Permission Denied! Running live capture requires Administrative or root privileges.")
except Exception as e:
    print(f"[!] Error occurred: {e}")